# Time-Series Forecasting: Cabai Merah Keriting (Red Curly Chili) Prices

Region: DKI Jakarta — Pasar Jatinegara, Pasar Kramatjati, Pasar Minggu  
Period: 01 June 2022 to 30 June 2026 (weekly)  
Source: `price.xlsx`

Weekly price-forecasting with pandas + statsmodels (convertdate for Ramadan/Eid features).

Steps: load and reshape, clean (gap weeks flagged), EDA, models (naive, drift, seasonal-naive, moving average, Holt-Winters/ETS, SARIMAX+Ramadan), rolling-origin backtest at horizons 1/2/4 weeks, final forecast with 95% intervals.

Notes:
- Selection and scores use a rolling-origin backtest, not one 12-week holdout.
- Horizons kept short (1/2/4 weeks); long horizons are unreliable here.
- Originally-missing weeks are excluded from scoring.
- Simple persistence is often hard to beat. Where a baseline wins, it's kept.


## 0. Setup

Install dependencies. `openpyxl` reads the xlsx; `convertdate` gives the Ramadan/Eid dates.

In [ ]:
!pip install -q pandas statsmodels matplotlib openpyxl convertdate
print('Dependencies ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
pd.set_option('display.max_columns', 20)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('pandas', pd.__version__, '| numpy', np.__version__)

## 1. Load the data

On Colab, upload `price.xlsx` (or mount Drive and set `DATA_PATH`). Locally, point `DATA_PATH` at the file.

The sheet is wide: row 1 is weekly dates (`dd/ mm/ yyyy`), each row is a geographic level. Target markets are rows `a` Pasar Jatinegara, `b` Pasar Kramatjati, `c` Pasar Minggu. Prices use thousands commas (`"80,000"`) and `"-"` for missing weeks.

In [ ]:
# (Drive):
# from google.colab import drive; drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/price.xlsx'

# (local / default): file sits next to the notebook
DATA_PATH = 'price.xlsx'

raw = pd.read_excel(DATA_PATH, header=None)
print('Raw shape:', raw.shape)
raw.iloc[:7, :6]

## 2. Reshape to tidy long format

Select the three market rows, parse prices to numeric, melt into `[date, market, price]`.

In [ ]:
MARKET_LABELS = {
    'Pasar Jatinegara': 'Jatinegara',
    'Pasar Kramatjati': 'Kramatjati',
    'Pasar Minggu':     'Minggu',
}

def load_long(raw: pd.DataFrame) -> pd.DataFrame:
    date_strings = raw.iloc[0, 2:].astype(str).str.replace(' ', '', regex=False)
    dates = pd.to_datetime(date_strings, format='%d/%m/%Y')
    label_col = raw.iloc[:, 1].astype(str).str.strip()

    records = []
    for full_label, short in MARKET_LABELS.items():
        mask = label_col == full_label
        if not mask.any():
            raise ValueError(f'Market row not found: {full_label}')
        row = raw.loc[mask].iloc[0, 2:]
        prices = (row.astype(str)
                     .str.replace(',', '', regex=False)
                     .str.strip()
                     .replace({'-': np.nan, 'nan': np.nan, '': np.nan}))
        prices = pd.to_numeric(prices, errors='coerce')
        records.append(pd.DataFrame({'date': dates.values, 'market': short, 'price': prices.values}))
    long = pd.concat(records, ignore_index=True)
    return long.sort_values(['market', 'date']).reset_index(drop=True)

long = load_long(raw)
print(long.shape, '| range:', long['date'].min().date(), '->', long['date'].max().date())
print(long.groupby('market')['price'].agg(['count', 'min', 'max', 'mean']).round(0))
long.head()

## 3. Clean: weekly index + leak-aware gap handling

Put each market on a regular weekly index and interpolate the missing weeks. Keep `missing_mask` of the originally-missing weeks so they can be excluded from scoring (interpolation uses both neighbours, so a filled test value partly leaks the future).

In [ ]:
wide_raw = long.pivot(index='date', columns='market', values='price').sort_index()
full_idx = pd.date_range(wide_raw.index.min(), wide_raw.index.max(), freq='7D')
wide_raw = wide_raw.reindex(full_idx)
wide_raw.index.name = 'date'

# Mask of originally-missing weeks -> used to exclude from scoring
missing_mask = wide_raw.isna()
print('Originally-missing weeks per market:')
print(missing_mask.sum())

wide = wide_raw.interpolate(method='time').bfill().ffill()
print('\nMissing after fill:', int(wide.isna().sum().sum()), '| shape', wide.shape)
wide.head()

In [ ]:
ax = wide.plot(title='Weekly Cabai Merah Keriting Price by Market (Rp/kg)')
ax.set_ylabel('Price (Rp/kg)'); ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 4. Exploratory analysis

Chili prices are volatile and partly driven by Ramadan/Idul Fitri demand. The three markets are highly correlated, so from Section 5 one market (Jatinegara) is analysed in detail and the rest summarised.

In [ ]:
print('Correlation across markets:')
print(wide.corr().round(3))

monthly = wide.copy(); monthly['month'] = monthly.index.month
ax = monthly.groupby('month').mean().plot(marker='o', title='Average price by calendar month')
ax.set_xlabel('Month'); ax.set_ylabel('Price (Rp/kg)')
plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

SEASONAL_PERIOD = 52  # weekly data -> annual seasonality of around 52 weeks
DETAIL_MARKET = 'Jatinegara'
s = wide[DETAIL_MARKET].astype(float)

decomp = seasonal_decompose(s, period=SEASONAL_PERIOD, model='additive')
fig = decomp.plot(); fig.set_size_inches(12, 8)
fig.suptitle(f'Seasonal decomposition — {DETAIL_MARKET}', y=1.01)
plt.tight_layout(); plt.show()

adf_stat, adf_p, *_ = adfuller(s.dropna())
print(f'ADF test ({DETAIL_MARKET}): stat={adf_stat:.3f}, p-value={adf_p:.4f}')
# A non-rejection does NOT prove a unit root / trend exists.
if adf_p < 0.05:
    print('-> Rejects the unit-root null at 5%: consistent with (trend-)stationarity.')
else:
    print('-> Does not reject the unit-root null at 5%. This is inconclusive — it does')
    print('   not prove non-stationarity, but differencing is a reasonable precaution.')

### Ramadan / Idul Fitri features

The Islamic calendar shifts ~11 days earlier each year, so fixed 52-week seasonality blurs the Ramadan/Eid spike. Build two binary indicators (in-Ramadan week, within 10 days of Eid) for SARIMAX. Dates come from `convertdate`, so the same function covers future weeks.

In [ ]:
from convertdate import islamic

def ramadan_eid_frame(index: pd.DatetimeIndex) -> pd.DataFrame:
    """Binary weekly indicators: in-Ramadan week, and Eid-window week (+-10 days)."""
    ramadan = np.zeros(len(index)); eid = np.zeros(len(index))
    r_spans, e_days = [], []
    for gy in range(index.min().year - 1, index.max().year + 2):
        for iy in range(gy - 580, gy - 575):  # candidate Islamic years for this Gregorian year
            try:
                r1 = dt.date(*islamic.to_gregorian(iy, 9, 1))   # Ramadan 1
                e1 = dt.date(*islamic.to_gregorian(iy, 10, 1))  # Shawwal 1 = Eid al-Fitr
            except Exception:
                continue
            if r1.year == gy:
                r_spans.append((pd.Timestamp(r1), pd.Timestamp(r1) + pd.Timedelta(days=29)))
                e_days.append(pd.Timestamp(e1))
    for i, ts in enumerate(index):
        wk0, wk1 = ts, ts + pd.Timedelta(days=6)  # the week this observation covers
        if any(wk1 >= a and wk0 <= b for a, b in r_spans):
            ramadan[i] = 1
        if any(abs((ts - e).days) <= 10 for e in e_days):
            eid[i] = 1
    return pd.DataFrame({'ramadan': ramadan, 'eid': eid}, index=index)

exog_all = ramadan_eid_frame(wide.index)
print('Ramadan weeks flagged:', int(exog_all['ramadan'].sum()),
      '| Eid-window weeks:', int(exog_all['eid'].sum()))

# Visualise the spikes against the flags
fig, ax = plt.subplots()
ax.plot(s.index, s.values, color='steelblue', label=DETAIL_MARKET)
for d in exog_all.index[exog_all['eid'] == 1]:
    ax.axvline(d, color='orange', alpha=0.35)
ax.set_title('Price with Eid-window weeks highlighted (orange)')
ax.set_ylabel('Price (Rp/kg)'); ax.legend(); plt.tight_layout(); plt.show()

### Extended calendar features

Replaces `exog_all` with four flags: Ramadan, Eid al-Fitr, Eid al-Adha, and Christmas/New Year (Natal dan Tahun Baru / Nataru). Downstream SARIMAX cells pick these up automatically. Each flag needs at least one 1 in the training window or SARIMAX can't estimate it.

In [ ]:
def _islamic_dates(month, day, start, end, lo=1400, hi=1500):
    """All Gregorian dates in [start, end] on Islamic <month>/<day>."""
    out = []
    for iy in range(lo, hi):
        try:
            d = pd.Timestamp(dt.date(*islamic.to_gregorian(iy, month, day)))
        except Exception:
            continue
        if start <= d <= end:
            out.append(d)
    return out

def calendar_frame(index, eid_window=10, adha_window=7):
    """Weekly 0/1 flags on week-start dates.
    ramadan: week overlaps the ~30 days of Ramadan
    eid    : week start within +-eid_window days of Eid al-Fitr (1 Shawwal)
    adha   : week start within +-adha_window days of Eid al-Adha (10 Dhu al-Hijjah)
    nataru : week overlaps 20 Dec - 2 Jan"""
    start = index.min() - pd.Timedelta(days=60)
    end = index.max() + pd.Timedelta(days=60)
    ramadan_starts = _islamic_dates(9, 1, start, end)
    fitr = _islamic_dates(10, 1, start, end)
    adha = _islamic_dates(12, 10, start, end)
    rows = []
    for ts in index:
        wk0, wk1 = ts, ts + pd.Timedelta(days=6)
        ram = any(wk1 >= a and wk0 <= a + pd.Timedelta(days=29) for a in ramadan_starts)
        eid = any(abs((ts - e).days) <= eid_window for e in fitr)
        ada = any(abs((ts - e).days) <= adha_window for e in adha)
        nat = any(wk1 >= pd.Timestamp(y, 12, 20) and wk0 <= pd.Timestamp(y + 1, 1, 2)
                  for y in range(ts.year - 1, ts.year + 1))
        rows.append((ram, eid, ada, nat))
    return pd.DataFrame(rows, index=index,
                        columns=['ramadan', 'eid', 'adha', 'nataru']).astype(float)

ramadan_eid_frame = calendar_frame
exog_all = calendar_frame(wide.index)
print('Weeks flagged per event (full history):')
print(exog_all.sum().astype(int).to_string())

## 5. Models and rolling-origin evaluation

Metrics exclude originally-missing weeks. The backtest refits at many cut points and averages the errors.

In [ ]:
def metrics(y_true, y_pred, mask=None):
    """MAE/RMSE/MAPE. If mask is given, originally-missing weeks are dropped first."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    if mask is not None:
        keep = ~np.asarray(mask, bool)
        y_true, y_pred = y_true[keep], y_pred[keep]
    if len(y_true) == 0:
        return {'MAE': np.nan, 'RMSE': np.nan, 'MAPE%': np.nan}
    return {'MAE': np.mean(np.abs(y_true - y_pred)),
            'RMSE': np.sqrt(np.mean((y_true - y_pred) ** 2)),
            'MAPE%': np.mean(np.abs((y_true - y_pred) / y_true)) * 100}

def rolling_backtest(series, forecaster, miss, horizon=4, n_origins=20, step=4, min_train=104):
    """Average metrics over many forecast origins (newest first)."""
    scores = []
    end = len(series) - horizon
    for k in range(n_origins):
        cut = end - k * step
        if cut < min_train:
            break
        tr = series.iloc[:cut]
        te = series.iloc[cut:cut + horizon]
        te_mask = miss.iloc[cut:cut + horizon].values
        exog_future = exog_all.iloc[cut:cut + horizon]
        try:
            pred = forecaster(tr, horizon, exog_future)
        except TypeError:
            pred = forecaster(tr, horizon)
        scores.append(metrics(te.values, pred, te_mask))
    return pd.DataFrame(scores).mean()

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Each forecaster: (train_series, horizon[, exog_future]) -> np.ndarray of length horizon
def f_naive(tr, h, exf=None):
    return np.repeat(tr.values[-1], h)

def f_drift(tr, h, exf=None):
    """Last value + average weekly change (a simple trend baseline)."""
    v = tr.values; slope = (v[-1] - v[0]) / (len(v) - 1)
    return v[-1] + slope * np.arange(1, h + 1)

def f_snaive(tr, h, exf=None, p=SEASONAL_PERIOD):
    hist = tr.values
    return np.array([hist[-p + (i % p)] for i in range(h)])

def f_ma(tr, h, exf=None, w=4):
    return np.repeat(tr.values[-w:].mean(), h)

def f_ets(tr, h, exf=None, p=SEASONAL_PERIOD):
    tr = tr.astype(float)
    seasonal = 'add' if len(tr) >= 2 * p else None
    m = ETSModel(tr, trend='add', seasonal=seasonal,
                 seasonal_periods=p if seasonal else None, error='add').fit(disp=False)
    return np.asarray(m.forecast(h))

def f_sarimax(tr, h, exf, order=(1, 1, 1)):
    tr = tr.astype(float)
    m = SARIMAX(tr, exog=exog_all.loc[tr.index], order=order,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    return np.asarray(m.forecast(steps=h, exog=exf))

FORECASTERS = {
    'Naive': f_naive,
    'Drift': f_drift,
    'Seasonal-naive': f_snaive,
    'Moving-average': f_ma,
    'ETS(Holt-Winters)': f_ets,
    'SARIMAX+Ramadan': f_sarimax,
}
print('Models:', list(FORECASTERS))

### Skill-vs-naive evaluation

Scores each model against the naive baseline on identical windows. skill_vs_naive% > 0 beats naive, MASE < 1 beats the in-sample one-step naive, and gain_t is a rough paired t-stat (|t| < 2 ~ noise; origins overlap so it overstates confidence).

In [ ]:
def mase_scale(train):
    """Mean absolute one-step naive error on the training data (MASE denominator)."""
    return float(np.mean(np.abs(np.diff(np.asarray(train, float)))))

def rolling_eval(series, forecaster, miss, horizon=4, n_origins=40, step=1, min_train=104):
    """Per-origin errors for forecaster AND the naive baseline on identical windows."""
    rows = []
    end = len(series) - horizon
    for k in range(n_origins):
        cut = end - k * step
        if cut < min_train:
            break
        tr = series.iloc[:cut]
        y = series.iloc[cut:cut + horizon].values
        keep = ~miss.iloc[cut:cut + horizon].values
        if not keep.any():
            continue
        pred = np.asarray(forecaster(tr, horizon, exog_all.iloc[cut:cut + horizon]), float)
        naive = np.repeat(tr.values[-1], horizon)
        rows.append({
            'origin': tr.index[-1],
            'mae': np.mean(np.abs(y - pred)[keep]),
            'mae_naive': np.mean(np.abs(y - naive)[keep]),
            'mape': np.mean(np.abs((y - pred) / y)[keep]) * 100,
            'scale': mase_scale(tr),
        })
    return pd.DataFrame(rows)

def summarize_eval(df):
    d = df['mae_naive'] - df['mae']  # >0 means the model beat naive at that origin
    se = d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
    t = d.mean() / se if se and se > 0 else np.nan
    return pd.Series({
        'MAE': df['mae'].mean(),
        'MAPE%': df['mape'].mean(),
        'MASE': (df['mae'] / df['scale']).mean(),
        'skill_vs_naive%': 100 * (1 - df['mae'].mean() / df['mae_naive'].mean()),
        'gain_t': t,
        'n_origins': len(df),
    })

def evaluate_all(series, miss, forecasters, horizons=(1, 2, 4),
                 step_by_h=None, n_origins=40, min_train=104):
    """Long table: one row per (horizon, model). step=1 for short horizons."""
    step_by_h = step_by_h or {1: 1, 2: 1, 4: 2}
    out = []
    for h in horizons:
        for name, fc in forecasters.items():
            df = rolling_eval(series, fc, miss, horizon=h, n_origins=n_origins,
                              step=step_by_h.get(h, h), min_train=min_train)
            out.append({'horizon': h, 'model': name, **summarize_eval(df).to_dict()})
    return pd.DataFrame(out)

### Interval models + empirical coverage

ETS and SARIMAX with prediction intervals (damped trend, optional log). `coverage_backtest` checks how often the 95% interval contains the actual, and the mean width as a share of price.

In [ ]:
def pi_ets(tr, h, exf=None, alpha=0.05, log=False, damped=True, seasonal=None,
           p=SEASONAL_PERIOD):
    """ETS with prediction interval. Damped, non-seasonal by default; log models log(price)."""
    y = np.log(tr.astype(float)) if log else tr.astype(float)
    m = ETSModel(y, error='add', trend='add', damped_trend=damped,
                 seasonal=seasonal, seasonal_periods=p if seasonal else None).fit(disp=False)
    sf = m.get_prediction(start=len(y), end=len(y) + h - 1).summary_frame(alpha=alpha)
    mean, lo, hi = sf['mean'].values, sf['pi_lower'].values, sf['pi_upper'].values
    return (np.exp(mean), np.exp(lo), np.exp(hi)) if log else (mean, lo, hi)

def pi_sarimax(tr, h, exf, alpha=0.05, log=False, order=(1, 1, 1)):
    """SARIMAX with calendar regressors and a prediction interval."""
    y = np.log(tr.astype(float)) if log else tr.astype(float)
    m = SARIMAX(y, exog=exog_all.loc[tr.index], order=order,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    sf = m.get_forecast(steps=h, exog=exf).summary_frame(alpha=alpha)
    mean, lo, hi = sf['mean'].values, sf['mean_ci_lower'].values, sf['mean_ci_upper'].values
    return (np.exp(mean), np.exp(lo), np.exp(hi)) if log else (mean, lo, hi)

def coverage_backtest(series, pi_fn, miss, horizon=4, n_origins=40, step=2, min_train=104):
    """How often the interval contains the actual, plus mean width as % of price."""
    recs = []
    end = len(series) - horizon
    for k in range(n_origins):
        cut = end - k * step
        if cut < min_train:
            break
        tr = series.iloc[:cut]
        y = series.iloc[cut:cut + horizon].values
        gap = miss.iloc[cut:cut + horizon].values
        _, lo, hi = pi_fn(tr, horizon, exog_all.iloc[cut:cut + horizon])
        for j in range(horizon):
            if gap[j]:
                continue
            recs.append({'step': j + 1,
                         'covered': float(lo[j] <= y[j] <= hi[j]),
                         'width_pct': (hi[j] - lo[j]) / y[j] * 100})
    df = pd.DataFrame(recs)
    by_step = df.groupby('step')[['covered', 'width_pct']].mean()
    overall = {'coverage%': df['covered'].mean() * 100,
               'width%': df['width_pct'].mean(), 'n_points': len(df)}
    return overall, by_step

### Run the extended evaluation (detailed market)

Slow: SARIMAX refits at every origin.

In [ ]:
s = wide[DETAIL_MARKET].astype(float)
miss = missing_mask[DETAIL_MARKET]

# (a) point-forecast skill vs naive
ev = evaluate_all(s, miss, FORECASTERS)
print(f'Skill vs naive (%), {DETAIL_MARKET} -- higher is better:')
display(ev.pivot(index='model', columns='horizon', values='skill_vs_naive%').round(1))
print('Rough paired t-stat of the gain over naive (|t| < 2 ~ noise):')
display(ev.pivot(index='model', columns='horizon', values='gain_t').round(2))
print('MASE (< 1 beats in-sample one-step naive):')
display(ev.pivot(index='model', columns='horizon', values='MASE').round(2))

# (b) are the 95% intervals honest?
configs = {
    'ETS damped':        lambda tr, h, exf: pi_ets(tr, h, exf),
    'ETS damped (log)':  lambda tr, h, exf: pi_ets(tr, h, exf, log=True),
    'SARIMAX+cal':       lambda tr, h, exf: pi_sarimax(tr, h, exf),
    'SARIMAX+cal (log)': lambda tr, h, exf: pi_sarimax(tr, h, exf, log=True),
}
cov_rows = []
for name, fn in configs.items():
    overall, by_step = coverage_backtest(s, fn, miss, horizon=4)
    cov_rows.append({'model': name, **overall})
    print(f'{name} -- coverage by forecast step:'); display(by_step.round(1))
print('Nominal coverage is 95%. Far below = intervals too narrow (overconfident).')
display(pd.DataFrame(cov_rows).set_index('model').round(1))

### 5a. Error vs horizon (detailed market)

Backtest every model at horizons 1, 2, 4 weeks. SARIMAX refits at every origin, so this cell takes a minute or two.

In [ ]:
s = wide[DETAIL_MARKET].astype(float)
miss = missing_mask[DETAIL_MARKET]

rows = []
for h in (1, 2, 4):
    for name, fc in FORECASTERS.items():
        sc = rolling_backtest(s, fc, miss, horizon=h, n_origins=20, step=4)
        rows.append({'horizon': h, 'model': name, 'MAPE%': sc['MAPE%'], 'MAE': sc['MAE']})
hz = pd.DataFrame(rows)

pivot_mape = hz.pivot(index='model', columns='horizon', values='MAPE%').round(2)
print(f'Rolling-backtest MAPE% by horizon — {DETAIL_MARKET} (lower = better):')
display(pivot_mape)

ax = pivot_mape.T.plot(marker='o', title=f'Error growth with horizon — {DETAIL_MARKET}')
ax.set_xlabel('Forecast horizon (weeks)'); ax.set_ylabel('MAPE %'); ax.set_xticks([1, 2, 4])
plt.tight_layout(); plt.show()

Simple persistence is often hard to beat, and error grows with horizon. Holt-Winters and seasonal-naive estimate 52 seasonal values from ~4 cycles and may lose to a one-liner. SARIMAX+Ramadan helps mainly around those weeks, so averaged gains can be small.

### 5b. Model selection across markets

Pick the best model per market by rolling-backtest MAPE at h=4.

In [ ]:
SELECT_HORIZON = 4
sel_rows = []
for mkt in wide.columns:
    sm = wide[mkt].astype(float); mm = missing_mask[mkt]
    for name, fc in FORECASTERS.items():
        sc = rolling_backtest(sm, fc, mm, horizon=SELECT_HORIZON, n_origins=20, step=4)
        sel_rows.append({'market': mkt, 'model': name,
                         'MAPE%': sc['MAPE%'], 'MAE': sc['MAE']})
scoreboard = pd.DataFrame(sel_rows)
print(f'Rolling-backtest scoreboard (h={SELECT_HORIZON}, lower = better):')
display(scoreboard.round(2))

best = (scoreboard.sort_values('MAPE%')
                  .groupby('market', as_index=False)
                  .first()[['market', 'model', 'MAPE%']])
print('Best model per market (by rolling-backtest MAPE):')
display(best.round(2))

## 6. Final forecast with prediction intervals

Refit on full history and forecast `HORIZON=4` weeks ahead with 95% intervals, comparing two interval-producing models for the detailed market: ETS and SARIMAX+Ramadan.

In [ ]:
HORIZON = 4
step = pd.Timedelta('7D')
s = wide[DETAIL_MARKET].astype(float)
future_index = pd.date_range(s.index[-1] + step, periods=HORIZON, freq='7D')
exog_future = ramadan_eid_frame(future_index)

# ETS with 95% prediction interval
ets_fit = ETSModel(s, trend='add', seasonal='add',
                   seasonal_periods=SEASONAL_PERIOD, error='add').fit(disp=False)
ets_pred = ets_fit.get_prediction(start=len(s), end=len(s) + HORIZON - 1).summary_frame(alpha=0.05)
ets_fc = pd.DataFrame({'forecast': ets_pred['mean'].values,
                       'lower': ets_pred['pi_lower'].values,
                       'upper': ets_pred['pi_upper'].values}, index=future_index)

# SARIMAX + Ramadan/Eid with 95% interval
sx_fit = SARIMAX(s, exog=exog_all.loc[s.index], order=(1, 1, 1),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sx_pred = sx_fit.get_forecast(steps=HORIZON, exog=exog_future).summary_frame(alpha=0.05)
sx_fc = pd.DataFrame({'forecast': sx_pred['mean'].values,
                      'lower': sx_pred['mean_ci_lower'].values,
                      'upper': sx_pred['mean_ci_upper'].values}, index=future_index)

print(f'ETS forecast — {DETAIL_MARKET}:');     display(ets_fc.round(0))
print(f'SARIMAX+Ramadan forecast — {DETAIL_MARKET}:'); display(sx_fc.round(0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (title, fc) in zip(axes, [('ETS', ets_fc), ('SARIMAX+Ramadan', sx_fc)]):
    ax.plot(s.index[-52:], s.values[-52:], color='steelblue', label='history')
    ax.plot(fc.index, fc['forecast'], 'r--', marker='o', ms=5, label='forecast')
    ax.fill_between(fc.index, fc['lower'], fc['upper'], color='red', alpha=0.15, label='95% PI')
    ax.set_title(f'{DETAIL_MARKET} — {title} ({HORIZON} weeks)')
    ax.set_ylabel('Price (Rp/kg)'); ax.legend()
plt.tight_layout(); plt.show()

## 7. Export forecasts

Refit the model chosen in 5b per market and export to CSV. ETS and SARIMAX carry 95% intervals; the baselines are point-only.

In [ ]:
best_model_by_market = dict(zip(best['market'], best['model']))
print('Chosen models:', best_model_by_market)

def forecast_with_pi(series, model_name, horizon=HORIZON):
    """Return a DataFrame [forecast, lower, upper] (lower/upper NaN for point-only models)."""
    series = series.astype(float)
    fut = pd.date_range(series.index[-1] + step, periods=horizon, freq='7D')
    exf = ramadan_eid_frame(fut)
    if model_name == 'ETS(Holt-Winters)':
        fit = ETSModel(series, trend='add', seasonal='add',
                       seasonal_periods=SEASONAL_PERIOD, error='add').fit(disp=False)
        p = fit.get_prediction(start=len(series), end=len(series) + horizon - 1).summary_frame(alpha=0.05)
        return pd.DataFrame({'forecast': p['mean'].values, 'lower': p['pi_lower'].values,
                             'upper': p['pi_upper'].values}, index=fut)
    if model_name == 'SARIMAX+Ramadan':
        fit = SARIMAX(series, exog=exog_all.loc[series.index], order=(1, 1, 1),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        p = fit.get_forecast(steps=horizon, exog=exf).summary_frame(alpha=0.05)
        return pd.DataFrame({'forecast': p['mean'].values, 'lower': p['mean_ci_lower'].values,
                             'upper': p['mean_ci_upper'].values}, index=fut)
    # point-only baselines
    point = FORECASTERS[model_name](series, horizon, exf)
    return pd.DataFrame({'forecast': np.asarray(point), 'lower': np.nan, 'upper': np.nan}, index=fut)

export = []
for mkt in wide.columns:
    fdf = forecast_with_pi(wide[mkt], best_model_by_market[mkt]).reset_index()
    fdf = fdf.rename(columns={'index': 'date'})
    fdf.insert(1, 'market', mkt)
    fdf.insert(2, 'model', best_model_by_market[mkt])
    export.append(fdf)
export_df = pd.concat(export, ignore_index=True)
export_df['date'] = pd.to_datetime(export_df['date']).dt.date

out_path = 'chili_price_forecast.csv'
export_df.to_csv(out_path, index=False)
print(f'Saved {len(export_df)} rows -> {out_path}')
display(export_df.round(0))

---
### Notes

- Evaluation is rolling-origin, not a single holdout.
- Horizons are short (1/2/4 weeks); longer ones are unreliable.
- Simple baselines are strong; where one wins, it's kept.
- Ramadan/Eid are SARIMAX regressors; gains concentrate around those weeks.
- Originally-missing weeks are excluded from every metric.
- ETS and SARIMAX give 95% intervals; baselines are point-only.
- One market is detailed, the rest summarised.
